In [5]:
from dotenv import load_dotenv

load_dotenv()

True

In [6]:
SYSTEM_PROMPT = """你是一个专业的代码审查专家。对提供的代码进行分析，输出一份结构化评审报告，涵盖以下五个维度：

1. **Bug 与正确性** — 逻辑错误、边界情况、异常处理
2. **安全问题** — 注入风险、密钥泄露、不安全的操作
3. **性能问题** — 效率低下、不必要的计算、内存占用
4. **代码风格** — PEP 8 违规、命名规范、可读性
5. **改进建议** — 重构建议、更好的设计模式

格式：使用 Markdown。给出总体质量评级:🟢 优秀 / 🟡 需改进 / 🔴 存在严重问题"""

In [7]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

import os

model = os.getenv("MODEL_NAME")

def review_code(code: str, language: str = "python") -> str:
    if model is None:
        raise ValueError("MODEL_NAME is not set")
    llm = ChatOpenAI(model=model, temperature=0, extra_body={"enable_thinking": False})
    messages = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=f"Review 这份 {language} 代码:\n\n```{language}\n{code}\n```"),
    ]
    content = ''
    response = llm.stream(messages)
    for chunk in response:
        str = getattr(chunk, 'content')
        content += str
        print(str, end='', flush=True)

    return content

In [8]:
with open('/Users/linqibin/Documents/code/agents/my-agents/apps/backend/agents/research_agent.ipynb') as f:
    code = f.read()
    review_code(code=code, language='python')

# 代码审查报告

**总体质量评级:** 🟡 **需改进**

这份代码实现了一个基于 LangGraph 的简单研究代理（Research Agent），能够搜索网络并生成报告。虽然核心逻辑可行，但在**状态管理、错误处理、资源效率和安全性**方面存在显著缺陷，直接用于生产环境可能会导致数据丢失、API 浪费或安全风险。

---

### 1. Bug 与正确性 (Bug & Correctness)

*   **状态更新不完整 (Critical)**:
    *   在 `search_web` 函数中，返回了 `{"search_results": results}`。根据 LangGraph 的机制，这只会更新 `search_results` 字段，而不会保留之前的 `messages` 或其他状态。虽然在这个线性流程中可能看似没问题，但如果后续节点依赖完整的 State 对象，或者未来扩展为循环结构，这种部分更新可能导致意外行为。更安全的做法是返回包含所有必要字段的字典，或者确保 State 定义中的默认值处理得当。
    *   在 `synthesize_report` 中，返回了 `{"report": response.content, "messages": [response]}`。这里有一个潜在的逻辑错误：`messages` 被覆盖为只包含当前 LLM 响应的列表。如果上游节点（如 `search_web`）没有向 `messages` 添加任何内容（它确实没有），那么最终状态中的 `messages` 将丢失用户查询的历史记录。通常我们希望 `messages` 累积整个对话历史。
*   **缺少输入验证**:
    *   `search_web` 和 `synthesize_report` 都假设 `state["query"]` 和 `state["search_results"]` 存在且格式正确。如果传入的初始状态缺少这些键，程序会抛出 `KeyError`。
*   **硬编码的截断长度**:
    *   `[:500]` 硬编码在字符串切片中。如果搜索结果非常长，可能会截断关键信息；如果很短，则浪费 token。建议作为配置参数。

### 2. 安全问题 (Security)

*   **环境变量未